# OracleSchemaComp - interactive row-count comparison

Runs `src/rowcount_compare.py` (the same CLI used from a terminal/CI) and loads its
CSV reports into pandas DataFrames for quick inspection.

Requires `pandas` (see `requirements.txt`) and a real Oracle connection - unlike
`tests/` and `runbooks/`, these cells hit live databases. See the main
[README](../README.md) for full CLI docs, schema-resolution rules, and privilege
requirements.

In [ ]:
import sys
sys.path.insert(0, ".")  # so `import helper` works when the kernel's cwd is notebooks/

import helper
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

## Configure connections

Loads `../.env` into the kernel's environment (same variables the CLI reads:
`DB_A_USERNAME`/`DB_A_PASSWORD`/`DB_A_DSN`/`DB_A_SCHEMA`, and the `DB_B_*` equivalents).
Copy `.env.example` to `.env` and fill it in first if you haven't already.

Variables already set in the environment (e.g. by the kernel, or exported before
launching Jupyter) are left untouched - the file only fills in what's missing.

In [ ]:
helper.load_env_file()  # defaults to the project's .env

## Example 1: a single table

`--table` needs no table-list file. Both sides use their own connection's default
schema, since no `db_a_schema`/`db_b_schema` is given.

In [ ]:
result = helper.run_comparison(table="HR.EMPLOYEES")
print(f"exit code: {result.exit_code}")
result.report

## Example 2: a whole table list

In [ ]:
result = helper.run_comparison(tables_file="../tables.example.txt")
print(f"exit code: {result.exit_code}")
result.report

## Example 3: different schema per side

Same logical tables, but they live under a different owner in each environment
(e.g. comparing `HR_PROD` to `HR_UAT`). `db_a_schema`/`db_b_schema` supply the
default owner for any *unqualified* table name in the list; an explicit
`OWNER.TABLE` entry in the file (like `SALES.ORDERS` in `tables.example.txt`)
always overrides both. See the README's "Schema resolution" section for the
full priority order.

In [ ]:
result = helper.run_comparison(
    tables_file="../tables.example.txt",
    db_a_schema="HR_PROD",
    db_b_schema="HR_UAT",
)
print(f"exit code: {result.exit_code}")
result.report

Only the mismatched/errored rows, if any (`None` when everything matched):

In [ ]:
result.differences

## Reloading a previous run

Re-reads the latest CSVs from an `output_dir` without re-querying either database -
useful for revisiting a CI run's output, or a comparison someone else kicked off.

In [ ]:
report, differences = helper.load_latest_reports("../reports")
report